# Proyecto Fashion Peaks: 03 - Evaluación Comparativa y Métricas
En este notebook realizamos la evaluación integral y comparativa de los diferentes modelos:
1. **Modelo Original del Profesor** (`modelo_entrenado.pth`).
2. **Nuevo Modelo de Clasificación** (`modelo_clasificacion_mejorado.pth`).
3. **Nuevo Modelo de Regresión Continua** (`modelo_regresion_mejorado.pth`).

El objetivo es cuantificar si el sombreado del área (`v2`) y la formulación como regresión (`v3`) resuelven el problema de confusión observado en el rango de áreas de $9250$ a $17250$.


In [ ]:
# 1. Conexión de entorno y Google Drive
import sys, os

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    PATH_DRIVE = '/content/gdrive/MyDrive/Fashion_Peaks_Colab'
    if os.path.exists(PATH_DRIVE):
        os.chdir(PATH_DRIVE)
    print(f"-> Conectado a Google Colab. Directorio: {os.getcwd()}")
else:
    print(f"-> Entorno local. Directorio: {os.getcwd()}")


In [ ]:
# 2. Importaciones
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, mean_absolute_error
from tqdm import tqdm


In [ ]:
# 3. Definición de Arquitecturas
class SpectraCNN(nn.Module):
    def __init__(self, n_classes=40):
        super(SpectraCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class SpectraCNNRegressor(nn.Module):
    def __init__(self):
        super(SpectraCNNRegressor, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


In [ ]:
# 4. Inspección de Modelos Disponibles
rutas_modelos = {
    "Original Profesor (41 clases)": "modelos_guardados/modelo_original_profesor.pth",
    "Nuevo Clasificación (40 clases)": "modelos_guardados/modelo_clasificacion_mejorado.pth",
    "Nuevo Regresión Continua": "modelos_guardados/modelo_regresion_mejorado.pth"
}

for nombre, ruta in rutas_modelos.items():
    estado = "EXISTE" if os.path.exists(ruta) else "NO ENCONTRADO (Pendiente de entrenar)"
    print(f"- {nombre}: {ruta} -> [{estado}]")


In [ ]:
# 5. Función de Inferencia de un Espectro Individual
transform = transforms.Compose([transforms.ToTensor()])

def predecir_muestra(img_path, modelo, tipo='clasificacion', n_classes=40):
    modelo.eval()
    im = Image.open(img_path).convert('L')
    t = transform(im).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = modelo(t)
        if tipo == 'clasificacion':
            pred_clase = torch.argmax(out, dim=1).item()
            pred_area = pred_clase * 500 + 250
        else:
            pred_area = float(out.item()) * 20000.0
    return pred_area

# Ejemplo con la primera imagen
ejemplo_img = "imagenes_entrenamiento/spectro_00000.png"
if os.path.exists(ejemplo_img):
    plt.figure(figsize=(3, 3))
    plt.imshow(Image.open(ejemplo_img), cmap='gray')
    plt.title(f"Muestra: {ejemplo_img}")
    plt.axis('off')
    plt.show()


In [ ]:
# 6. Evaluación Masiva del Modelo de Regresión sobre el CSV
csv_reg = "imagenes_entrenamiento_regresion/labels.csv"
ruta_reg = "modelos_guardados/modelo_regresion_mejorado.pth"

if os.path.exists(csv_reg) and os.path.exists(ruta_reg):
    reg_model = SpectraCNNRegressor().to(device)
    reg_model.load_state_dict(torch.load(ruta_reg, map_location=device))
    reg_model.eval()
    
    df_reg = pd.read_csv(csv_reg)
    reales, preds = [], []
    
    print(f"Evaluando {len(df_reg)} muestras en el modelo de regresión...")
    for idx, row in tqdm(df_reg.iterrows(), total=len(df_reg)):
        img_p = os.path.join("imagenes_entrenamiento_regresion", row.iloc[0])
        val_real = float(row.iloc[1])
        val_pred = predecir_muestra(img_p, reg_model, tipo='regresion')
        
        reales.append(val_real)
        preds.append(val_pred)
        
    reales = np.array(reales)
    preds = np.array(preds)
    
    mae_global = mean_absolute_error(reales, preds)
    print(f"-> MAE Global de Regresión: {mae_global:.2f} unidades de área")
    
    # Análisis en la zona crítica (9250 a 17250)
    mask = (reales >= 9250) & (reales <= 17250)
    mae_critico = mean_absolute_error(reales[mask], preds[mask])
    print(f"-> MAE en Región Crítica [9250 - 17250]: {mae_critico:.2f} unidades de área")
    
    plt.figure(figsize=(8, 6))
    plt.scatter(reales, preds, alpha=0.4, color='navy', s=15, label='Espectros')
    plt.plot([0, 20000], [0, 20000], 'r--', label='Ideal y=x')
    plt.axvspan(9250, 17250, color='orange', alpha=0.2, label='Zona Crítica (9250 - 17250)')
    plt.xlabel('Área Real')
    plt.ylabel('Área Predicha')
    plt.title('Rendimiento de Regresión en todo el espectro')
    plt.legend(); plt.grid(True)
    plt.show()
else:
    print("El modelo de regresión aún no ha sido entrenado. Ejecutá primero '02_nuevo_entrenamiento_regresion.ipynb'.")
